In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)


# 📚 第三周-Day1：预训练——数据、规模与涌现能力

> **万丈高楼平地起。在ChatGPT问世之前，没有人相信"读完全互联网"就能诞生智能。但GPT-3用1750亿参数证明了一件事：量变，真的能引起质变。今天，我们走进大模型训练的起点——预训练。**


## 📅 学习进度

| 阶段 | 状态 |
|------|------|
| W1：Transformer 基础架构 | ✅ 已完成 |
| W2：Transformer 深入理解 | ✅ 已完成 |
| **W3：大模型训练全景** | 🔄 **进行中（Day 1/7）** |
| W4-W12 | ⏳ 待开始 |

---


## 一、为什么需要预训练？

### 1.1 从"教孩子说话"说起

想象你要教一个孩子学中文。有两种方法：

- **方法A**：一本字典都不给，直接让他做语文考试——结果可想而知
- **方法B**：先让他读一万本书，自然学会语感和常识，再去做题

预训练就是**方法B**。

在大模型的世界里，预训练是指：让模型在海量无标注文本上自主学习语言规律、世界知识和推理能力的阶段。这个阶段不需要人手把手教，而是让模型通过"预测下一个词"这个极简任务，自然而然地吸收知识。

### 1.2 预训练前后的本质区别

| 维度 | 预训练前 | 预训练后 |
|------|---------|---------|
| 语言能力 | 只懂随机字符 | 语法正确、语义通顺 |
| 知识储备 | 零 | 包含百科、历史、科学等 |
| 推理能力 | 无 | 初步的因果和逻辑推理 |
| 能否对话 | ❌ 只会续写文本 | ⚠️ 能续写，但不会"回答问题" |

> **注意**：预训练后的模型还不能当聊天机器人用！它只是一个"语言大师"，你需要追问它才能触发回答。这就是为什么后面还需要 SFT 和 RLHF。

### 1.3 生活类比：预训练 = 读万卷书


In [ ]:
📚 预训练  ←→  一个人从出生到高中毕业的自主阅读
🎓 SFT    ←→  大学专业课训练（学会按指令做事）
🤝 RLHF   ←→  职场礼仪培训（学会说"得体"的话）
🚀 部署   ←→  正式上班！


---


## 二、核心原理详解

### 2.1 预训练的核心任务：Next Token Prediction

预训练的任务看起来极其简单——**预测下一个词**：


In [ ]:
输入: "今天天气真"
模型预测: "好" (概率 60%) / "热" (概率 25%) / "冷" (概率 10%) ...


但别小看这个任务。为了做好"预测下一个词"，模型必须学会：
- **语法规则**：主谓宾结构、时态、介词搭配
- **世界知识**："北京是中国的首都"
- **逻辑推理**："如果A>B, B>C, 那么A>C"
- **上下文理解**：前文提到了"猫"，后面用"它"指代

这就像一个学生为了应付"完形填空"考试，不得不把整本课本都学会了。

### 2.2 Scaling Laws：规模定律

2020年，OpenAI 发表了一篇改变 AI 历史的论文，揭示了模型性能与三个变量的数学关系：

**三个关键变量：**
1. **参数量 (N)**：模型的"脑容量"
2. **数据量 (D)**：训练用的 token 数
3. **算力 (C)**：训练用的 FLOPs（浮点运算量）

**核心公式（简化版）：**


In [ ]:
Loss ≈ A / N^α + B / D^β + C_loss

其中 α ≈ 0.076, β ≈ 0.095（经验值）


**说人话**：
- 参数翻倍 → 损失下降约 5%
- 数据翻倍 → 损失下降约 6%
- 两者都翻倍 → 效果叠加
- 但边际递减：从 1B→2B 的提升，远大于 100B→200B

**生活类比**：就像做饭——
- N = 锅的大小（太小了装不下）
- D = 食材的量（太少了吃不饱）
- C = 火候（太弱了煮不熟）
- 三者必须匹配！一口小锅配一吨食材 = 浪费

### 2.3 涌现能力：大模型的"魔法时刻"

**涌现 (Emergence)** 是复杂系统科学中的一个概念：整体出现了部分所没有的性质。

在 AI 领域，涌现表现为：**模型规模到达某个临界点后，突然获得了小模型完全没有的能力。**

| 能力 | <1B 参数 | 10B | 70B+ | 涌现原因 |
|------|---------|-----|------|---------|
| 算术运算 | ❌ 完全不会 | ⚠️ 简单加减 | ✅ 复杂计算 | 模型开始"理解"数学规律 |
| 代码生成 | ❌ | ⚠️ 只能补全 | ✅ 写完整程序 | 编程语言的模式识别 |
| 逻辑推理 | ❌ | ⚠️ 简单推理 | ✅ 多步推理 | 因果链建模 |
| 多语言翻译 | ❌ | ⚠️ 粗糙 | ✅ 信达雅 | 跨语言语义对齐 |
| 指令遵循 | ❌ | ⚠️ 勉强 | ✅ 精准执行 | 对指令格式的理解 |

> ⚠️ **重要提醒**：涌现不是"越大越好"的线性增长，而是"突然开窍"的相变。就像水温从 99°C 到 100°C，突然从液态变气态。

### 2.4 预训练数据 pipeline

工业级预训练数据处理是一条精密的流水线：


In [ ]:
🌐 互联网爬取 (Common Crawl, 45TB)
    ↓
🧹 规则清洗 (去重、过滤低质量、去HTML标签)
    ↓
🤖 模型清洗 (用小模型给数据打质量分)
    ↓
⚖️ 比例配比 (网页40% + 书籍20% + 论文15% + 代码15% + 对话10%)
    ↓
🔢 Tokenize (把文字变成数字序列)
    ↓
📦 打包成训练批次


**关键数据集举例：**
- **Common Crawl**：网页爬虫数据，量最大但质量参差
- **Books3 / BookCorpus**：书籍文本，质量高、知识密集
- **arXiv**：学术论文，逻辑推理丰富
- **GitHub**：代码数据，提升编程能力
- **Wikipedia**：百科知识，事实准确

---


## 三、代码实战

### 3.1 模拟 Scaling Law 可视化


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 设置中文字体
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

# 模拟 Scaling Law：参数量 vs Loss
parameters = np.logspace(8, 12, 50)  # 1亿到1万亿参数
loss_scaling = 2.5 * (parameters / 1e12) ** (-0.076) + 0.5

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(parameters, loss_scaling, 'b-o', linewidth=2, markersize=4)
ax.set_xlabel('参数量 (N)', fontsize=13)
ax.set_ylabel('验证集 Loss', fontsize=13)
ax.set_title('Scaling Law：模型越大，Loss越低', fontsize=15)

# 标注关键节点
for size, label in [(1e9, '1B'), (1e10, '10B'), (1e11, '100B'), (1e12, '1T')]:
    ax.axvline(x=size, color='gray', linestyle='--', alpha=0.3)
    ax.text(size, loss_scaling[np.argmin(np.abs(parameters - size))] * 1.1,
            label, fontsize=11, ha='center')

ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('scaling_law.png', dpi=150)
plt.show()
print("✅ Scaling Law 可视化完成！可以看出：参数量增大，Loss持续下降，但递减速度放慢。")


### 3.2 模拟涌现能力的相变曲线


In [ ]:
# 模拟不同模型规模下的"涌现"能力
model_sizes = np.array([0.5, 1, 2, 7, 13, 30, 70, 175, 540])  # 单位：B
# 算术能力：在10B以下几乎为零，之后突然跳升
arithmetic = np.where(model_sizes < 10, 
                       np.random.normal(2, 1, len(model_sizes)),
                       np.where(model_sizes < 70,
                                20 + (model_sizes - 10) * 1.5,
                                80 + (model_sizes - 70) * 0.2))
arithmetic = np.clip(arithmetic, 0, 95)

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogx(model_sizes, arithmetic, 'r-o', linewidth=2, markersize=8)
ax.axvspan(10, 70, alpha=0.15, color='orange', label='涌现过渡区')
ax.set_xlabel('模型参数量 (B)', fontsize=13)
ax.set_ylabel('算术准确率 (%)', fontsize=13)
ax.set_title('涌现能力：从"完全不会"到"突然开窍"', fontsize=15)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("✅ 注意曲线的拐点：不是线性增长，而是S形跃迁！")


### 3.3 预训练 Loss 下降曲线模拟


In [ ]:
# 模拟预训练过程中的 Loss 变化
steps = np.arange(1, 10001)
loss = 4.0 * np.exp(-steps / 2000) + 0.6 + 0.02 * np.random.randn(len(steps))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# 线性坐标
ax1.plot(steps, loss, 'b-', linewidth=0.8, alpha=0.7)
ax1.set_ylabel('Training Loss', fontsize=12)
ax1.set_title('预训练 Loss 曲线（线性坐标）', fontsize=14)
ax1.grid(True, alpha=0.3)

# 对数坐标
ax2.plot(steps, loss, 'b-', linewidth=0.8, alpha=0.7)
ax2.set_yscale('log')
ax2.set_xlabel('训练步数 (Steps)', fontsize=12)
ax2.set_ylabel('Training Loss (log)', fontsize=12)
ax2.set_title('预训练 Loss 曲线（对数坐标）', fontsize=14)
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()
print("✅ Loss 初期快速下降，后期趋于平稳。这就是为什么预训练后期越来越'贵'。")


---


## 四、可视化理解

### 4.1 数据配比饼图


In [ ]:
# 主流大模型的预训练数据配比（近似值）
labels = ['网页文本', '书籍', '学术论文', '代码', '对话数据', '其他']
sizes = [40, 20, 12, 15, 8, 5]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']
explode = (0.05, 0, 0, 0, 0, 0)  # 突出网页文本

fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(sizes, explode=explode, labels=labels,
                                    autopct='%1.0f%%', colors=colors,
                                    shadow=True, startangle=90)
ax.set_title('预训练数据来源分布（典型配比）', fontsize=15)
plt.tight_layout()
plt.show()


### 4.2 训练成本三角图


In [ ]:
# 参数量、数据量、算力的三角关系
fig, ax = plt.subplots(figsize=(8, 7))

# 三个角
corners = {'参数量 N': (0, 1), '数据量 D': (-0.866, -0.5), '算力 C': (0.866, -0.5)}
for name, (x, y) in corners.items():
    ax.plot(x, y, 'ro', markersize=15)
    ax.annotate(name, (x, y), textcoords='offset points',
                xytext=(0, 20 if y > 0 else -25), ha='center', fontsize=14, fontweight='bold')

# 三条边
triangle_x = [0, -0.866, 0.866, 0]
triangle_y = [1, -0.5, -0.5, 1]
ax.plot(triangle_x, triangle_y, 'b-', linewidth=2)

# 中心标注
ax.text(0, 0, 'Scaling\nLaw\n平衡', ha='center', va='center', fontsize=16,
        fontweight='bold', bbox=dict(boxstyle='round', facecolor='lightyellow'))

ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.2, 1.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('预训练的三要素平衡', fontsize=15)
plt.tight_layout()
plt.show()
print("三者必须匹配发展——任何一个短板都会拖垮整体效果。")


---


## 五、业务关联

### 5.1 和企业AI的关系

| 业务场景 | 预训练的关联 |
|---------|------------|
| **LangChat 智能客服** | 底层模型的选择决定了客服能力的上限 |
| **Agent 自主决策** | 模型规模影响推理和多步规划能力 |
| **RAG 知识增强** | 预训练模型的语言理解力是检索精度的基础 |
| **代码助手** | 模型的涌现能力直接决定代码质量 |

### 5.2 为什么企业不需要自己预训练？

- **成本极高**：训练 GPT-3 级别模型需要 ~460 万美元电费+算力
- **数据门槛**：高质量 TB 级数据的清洗 pipeline 极其复杂
- **工程难度**：千卡并行训练，一个节点崩溃就要重来
- **正确做法**：站在巨人的肩膀上——用开源预训练模型（Qwen、LLaMA、ChatGLM）做微调

### 5.3 企业选型建议


In [ ]:
预算 < 1万/月   → Qwen2-7B + LoRA 微调
预算 1-5万/月   → Qwen2-32B / Qwen2-72B API
预算 > 5万/月   → 考虑自建模型服务，或多模型路由


---


## 六、常见误区

### ❌ 误区1："参数越多模型越聪明"
**事实**：参数量只是条件之一。Llama-3 8B 可以超过很多早期的 70B 模型，因为数据质量和训练方法的进步。

### ❌ 误区2："涌现能力是玄学"
**事实**：涌现是有数学基础的相变现象。Stanford 2023 年的研究表明，某些"涌现"在合适的评估指标下其实是连续的。但不可否认，大模型确实获得了小模型完全不具备的能力。

### ❌ 误区3："预训练数据越多越好"
**事实**：**质量 > 数量**。重复数据会让模型"背题"，低质量数据会拉低整体表现。LIMA 论文证明：1000 条高质量数据胜过百万条低质量数据。

### ❌ 误区4："我们可以用企业数据从头预训练"
**事实**：企业数据通常只有 GB 级别，远不够预训练需要的 TB 级别。正确做法是：基于开源预训练模型做领域微调。

---


## 🧪 课堂练习（5分钟）

**练习1**：画一画你觉得"涌现能力"应该长什么样的曲线？（S形？阶跃形？指数形？）

**练习2**：如果你的公司有 100GB 的法律文书，你会选择：
- A. 从头预训练一个法律大模型
- B. 用 Qwen2-7B 做领域微调
- C. 直接用 GPT-4 API
选哪个？为什么？

**练习3**：一个模型在预训练 Loss = 2.3，另一个 Loss = 1.8。能说第二个一定比第一个好吗？为什么？

---


## 📝 课后测试（15分钟）

**❶** Scaling Law 中的三个关键变量是？
- A. CPU、内存、硬盘
- B. 参数量、数据量、算力
- C. 学习率、批次大小、训练轮数
- D. 模型深度、宽度、层数

**❷** 关于涌现能力，以下说法正确的是？
- A. 涌现能力可以通过增加训练轮数在小模型上获得
- B. 涌现能力只在模型规模超过某个临界点后出现
- C. 涌现能力是随机出现的，没有规律
- D. 所有能力都是涌现出来的

**❸** GPT-3 的参数量大约是？
- A. 1.5 亿
- B. 17.5 亿
- C. 175 亿
- D. 1750 亿

**❹** 以下哪个不是预训练数据的常见来源？
- A. Common Crawl
- B. Wikipedia
- C. 企业内部聊天记录
- D. GitHub 代码

**❺** 简答题：为什么说"预测下一个词"这个简单任务能让模型学到复杂的知识和推理能力？

---


## 🔑 今日术语

| 英文 | 音标 | 中文 |
|------|------|------|
| Pre-training | /ˈpriːˌtreɪnɪŋ/ | 预训练 |
| Scaling Law | /ˈskeɪlɪŋ lɔː/ | 规模定律 |
| Emergent Ability | /ɪˈmɜːrdʒənt əˈbɪlɪti/ | 涌现能力 |
| Next Token Prediction | /nɛkst ˈtoʊkən prɪˈdɪkʃən/ | 下一个词预测 |
| Tokenizer | /ˈtoʊkənaɪzər/ | 分词器 |
| FLOPs | /flɒps/ | 浮点运算次数 |
| Common Crawl | /ˈkɒmən krɔːl/ | 公共爬虫数据集 |
| Parameter | /pəˈræmɪtər/ | 参数 |

---


## 📎 参考资源

### 必读论文
1. 📄 **Scaling Laws for Neural Language Models** (Kaplan et al., 2020)
   - https://arxiv.org/abs/2001.08361
2. 📄 **Emergent Abilities of Large Language Models** (Wei et al., 2022)
   - https://arxiv.org/abs/2206.07682
3. 📄 **GPT-3: Language Models are Few-Shot Learners** (Brown et al., 2020)
   - https://arxiv.org/abs/2005.14165

### 视频推荐
1. 📺 **大模型训练原理详解**（B站，约25分钟）
   - https://www.bilibili.com/video/BV1w4411L7t3
2. 📺 **Scaling Law 可视化讲解**（B站，约15分钟）
   - https://www.bilibili.com/video/BV1qP4y1D7nA

### 明日预告
明天我们将进入 **SFT（监督微调）** 的世界——预训练模型学会了"说话"，但还不会"回答问题"。SFT 就是教它"听指令做事"的关键一步！🚀
